<a href="https://colab.research.google.com/github/sathiysn/BANA7075-Final/blob/main/BANA_7075_Final_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas scikit-learn xgboost mlflow joblib psycopg2-binary sqlalchemy fastapi uvicorn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/83

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, classification_report

from xgboost import XGBClassifier

import mlflow
import mlflow.sklearn

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from google.colab import userdata

DB_USER = "postgres.jtznwazvwqwrmorjwzpu"
DB_PASS = userdata.get('DB_PASS')
DB_HOST = "aws-1-us-west-2.pooler.supabase.com"
DB_PORT = "5432"
DB_NAME = "postgres"

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")


SecretNotFoundError: Secret DB_PASS does not exist.

In [ ]:
query = """
SELECT
    id,
    machine_id,
    timestamp,
    machine_type,
    install_year,
    shift,
    material,
    vib_mean,
    vib_std,
    vib_roc,
    vib_kurtosis,
    temp_mean,
    temp_std,
    temp_roc,
    temp_kurtosis,
    current_mean,
    current_std,
    current_roc,
    acoustic_mean,
    acoustic_std,
    acoustic_roc,
    oil_pressure_mean,
    oil_pressure_std,
    oil_pressure_roc,
    failure_within_4hrs,
    actual_failure,
    is_valid,
    ingested_at
FROM public.validated_features
WHERE is_valid = true;
"""

df = pd.read_sql(query, engine)
print(df.shape)
df.head()

In [ ]:
print("Target value counts:")
print(df["failure_within_4hrs"].value_counts(dropna=False))
print()

print("Target distribution:")
print(df["failure_within_4hrs"].value_counts(normalize=True, dropna=False))

# Define features and target

In [ ]:
feature_cols_numeric = [
    "install_year",
    "vib_mean", "vib_std", "vib_roc", "vib_kurtosis",
    "temp_mean", "temp_std", "temp_roc", "temp_kurtosis",
    "current_mean", "current_std", "current_roc",
    "acoustic_mean", "acoustic_std", "acoustic_roc",
    "oil_pressure_mean", "oil_pressure_std", "oil_pressure_roc"
]

feature_cols_categorical = [
    "machine_type",
    "shift",
    "material"
]

target_col = "failure_within_4hrs"

X = df[feature_cols_numeric + feature_cols_categorical].copy()
y = df[target_col].astype(int)

## One hot encode categorical variables (machine_type, shift, material)

In [ ]:
X = pd.get_dummies(
    X,
    columns=feature_cols_categorical,
    drop_first=True
)

print("Feature matrix shape:", X.shape)
print("First 10 columns:")
print(X.columns[:10].tolist())


## Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train class balance:")
print(y_train.value_counts(normalize=True))

# Setting up MLFlow

In [ ]:
mlflow.set_experiment("PMIS_MVP")
mlflow.sklearn.autolog(disable=True)

os.makedirs("artifacts", exist_ok=True)

# Train Random Forest and Log It

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42
)

with mlflow.start_run(run_name="RandomForest_v1"):
    rf.fit(X_train, y_train)

    rf_probs = rf.predict_proba(X_test)[:, 1]
    rf_preds = (rf_probs >= 0.5).astype(int)

    rf_recall = recall_score(y_test, rf_preds, zero_division=0)
    rf_precision = precision_score(y_test, rf_preds, zero_division=0)
    rf_f1 = f1_score(y_test, rf_preds, zero_division=0)
    rf_auc = roc_auc_score(y_test, rf_probs)

    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("feature_count", X_train.shape[1])

    mlflow.log_metric("recall", rf_recall)
    mlflow.log_metric("precision", rf_precision)
    mlflow.log_metric("f1", rf_f1)
    mlflow.log_metric("auc", rf_auc)

    rf_model_path = "artifacts/random_forest_v1.pkl"
    rf_feature_path = "artifacts/random_forest_features_v1.pkl"

    joblib.dump(rf, rf_model_path)
    joblib.dump(list(X_train.columns), rf_feature_path)

    mlflow.log_artifact(rf_model_path)
    mlflow.log_artifact(rf_feature_path)

print("Random Forest metrics")
print("Recall:", rf_recall)
print("Precision:", rf_precision)
print("F1:", rf_f1)
print("AUC:", rf_auc)
print()
print(classification_report(y_test, rf_preds, zero_division=0))

# Train XGBoost and log it

In [ ]:
pos_count = y_train.sum()
neg_count = len(y_train) - pos_count

if pos_count == 0:
    raise ValueError("No positive class found in y_train. You need more training data before modeling.")

scale_pos_weight = neg_count / pos_count

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

with mlflow.start_run(run_name="XGBoost_v1"):
    xgb.fit(X_train, y_train)

    xgb_probs = xgb.predict_proba(X_test)[:, 1]
    xgb_preds = (xgb_probs >= 0.5).astype(int)

    xgb_recall = recall_score(y_test, xgb_preds, zero_division=0)
    xgb_precision = precision_score(y_test, xgb_preds, zero_division=0)
    xgb_f1 = f1_score(y_test, xgb_preds, zero_division=0)
    xgb_auc = roc_auc_score(y_test, xgb_probs)

    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("scale_pos_weight", float(scale_pos_weight))
    mlflow.log_param("feature_count", X_train.shape[1])

    mlflow.log_metric("recall", xgb_recall)
    mlflow.log_metric("precision", xgb_precision)
    mlflow.log_metric("f1", xgb_f1)
    mlflow.log_metric("auc", xgb_auc)

    xgb_model_path = "artifacts/xgboost_v1.pkl"
    xgb_feature_path = "artifacts/xgboost_features_v1.pkl"

    joblib.dump(xgb, xgb_model_path)
    joblib.dump(list(X_train.columns), xgb_feature_path)

    mlflow.log_artifact(xgb_model_path)
    mlflow.log_artifact(xgb_feature_path)

print("XGBoost metrics")
print("Recall:", xgb_recall)
print("Precision:", xgb_precision)
print("F1:", xgb_f1)
print("AUC:", xgb_auc)
print()
print(classification_report(y_test, xgb_preds, zero_division=0))

# Compare the Models

In [ ]:
results = pd.DataFrame([
    {
        "model": "RandomForest",
        "recall": rf_recall,
        "precision": rf_precision,
        "f1": rf_f1,
        "auc": rf_auc,
        "artifact_path": rf_model_path,
        "feature_path": rf_feature_path
    },
    {
        "model": "XGBoost",
        "recall": xgb_recall,
        "precision": xgb_precision,
        "f1": xgb_f1,
        "auc": xgb_auc,
        "artifact_path": xgb_model_path,
        "feature_path": xgb_feature_path
    }
])

results.sort_values(by=["f1", "recall", "auc"], ascending=False)

In [ ]:
best_row = results.sort_values(by=["f1", "recall", "auc"], ascending=False).iloc[0]

best_model_type = best_row["model"]
best_artifact_path = best_row["artifact_path"]
best_feature_path = best_row["feature_path"]

best_model_version = (
    "random_forest_v1" if best_model_type == "RandomForest" else "xgboost_v1"
)

best_recall = float(best_row["recall"])
best_precision = float(best_row["precision"])
best_f1 = float(best_row["f1"])
best_auc = float(best_row["auc"])

print("Champion model:")
print(best_row)

In [ ]:
insert_sql = text("""
INSERT INTO public.model_registry (
    model_version,
    model_type,
    recall,
    precision,
    f1,
    auc,
    artifact_path,
    feature_path,
    status
)
VALUES (
    :model_version,
    :model_type,
    :recall,
    :precision,
    :f1,
    :auc,
    :artifact_path,
    :feature_path,
    :status
);
""")

with engine.begin() as conn:
    conn.execute(
        insert_sql,
        {
            "model_version": best_model_version,
            "model_type": best_model_type,
            "recall": best_recall,
            "precision": best_precision,
            "f1": best_f1,
            "auc": best_auc,
            "artifact_path": best_artifact_path,
            "feature_path": best_feature_path,
            "status": "champion"
        }
    )

print("Champion model inserted into model_registry.")